# 第六课｜什么是 RTL？

上一课已经能把判断写成 Boolean 关系。现在需要描述：有哪些输入输出、哪个值要保存、在什么时刻更新。

今天只解决：
> **怎样用代码一样的文字描述数字硬件，而不是描述 CPU 要逐行执行的程序？**

主要新概念：**寄存器传输级（Register-Transfer Level, RTL）**。


## 1. 本课使用的概念

第05课已解释时钟事件控制、同步复位和非阻塞赋值。本课学习 RTL 的设计层次、模块和端口，并用 Icarus Verilog 编译累加器。


## 2. 四个词逐个认识

**硬件描述语言（Hardware Description Language, HDL）**：描述数字硬件结构和行为的一类语言。

**SystemVerilog**：本项目采用的 HDL 与验证语言。

**RTL**：关注寄存器保存什么，以及数据如何在时钟周期之间计算和传递。

**module / port**：module 是有明确边界的硬件单元；port 是穿过这个边界的输入输出信号。


## 3. HDL 看起来像程序，但含义不同

Python 的相邻语句通常表示先后执行。RTL 的许多语句是在描述同时存在的硬件关系。读 RTL 时先问：输入是什么？输出是什么？哪个值是 state？哪个 clock edge 更新 state？


## 4. 只读一个已知行为：clocked accumulator

它和第四课的 accumulator 是同一个规则，只是换成 SystemVerilog：

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```


## 5. 从声明读出信号的含义

`module` 与 `endmodule` 定义模块；模块名 `clocked_accumulator` 由作者选择。`clk`、`rst_n`、`input_value` 是三个输入端口，`state` 是输出端口。单独的 `logic` 默认是一位，`logic signed [7:0]` 是八位有符号信号。

`always_ff`、`posedge`、`input`、`output` 都是标准关键字，`clk` 和 `state` 是自定义名称。`@(posedge clk)` 在时钟上升沿触发；`state <= state + input_value` 用旧状态求和并在当前时间步的非阻塞赋值阶段更新。第05课有逐项解释和边沿时序表。

八位补码只能表示 −128 到 127。本例 `127 + 1` 的低八位是 `10000000`，解释为 −128。编译工具可以接受这个运算；选用回绕还是饱和须由数值规则确定。


## 6. reset 也是 contract

`rst_n` 中 `_n` 表示 active-low：值为 0 时 reset 有效。这里是同步 reset——它只在 clock edge 上改变 state。reset polarity 与 timing 必须写清楚。


## 7. Run：编译模块

**Icarus Verilog** 是 HDL 编译与仿真工具。下面的 `iverilog` 命令使用 `-g2012` 选择 SystemVerilog 2012 语法，`-tnull` 检查语法并展开模块，不生成仿真程序。`subprocess.run(..., check=True)` 调用外部工具，并在工具返回失败时报告错误。

先预测：删除端口声明后的逗号，属于语法错误还是累加功能错误？本节保留原文件，只运行编译检查。


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
if not iverilog:
    print('Icarus Verilog not found; RTL compile check did not run.')
else:
    subprocess.run([iverilog, '-g2012', '-tnull', str(root/'rtl/learning/clocked_accumulator.sv')], check=True)
    print('COMPILE PASS: clocked_accumulator.sv')


## 8. Observe

`COMPILE PASS` 只说明语法和基本 elaboration 能通过，不说明 `1,2,3` 一定得到 `1,3,6`。这正好引出以后为什么需要 testbench。


## 9. Try It

标出三个输入端口和一个输出端口，再指出保存状态的信号、时钟事件和复位分支。手算 `state=3, input_value=2, rst_n=1` 的下一个上升沿结果，并解释编译成功为什么还未检查这个结果。


## 作业

[第 6 课作业：把一个 RTL 时钟沿翻译成 Python 语义](../../exercises/zh/06_what_is_rtl.ipynb)

这份独立作业检查本课的语义理解；课程中的真实 RTL / testbench 实验仍然保留。

## 10. AI Task

让 AI 只给现有 module 加注释，标出 module boundary、ports、stored state、clocked update；不改代码。


## 11. Human Check

不用 AI，你应该能解释 HDL 与普通软件的主要语义差别；module/port 各自解决什么问题；`always_ff @(posedge clk)` 告诉你什么；为什么 compile pass 不等于 behavior correct。


## 12. Engineering Handoff

`rtl/learning/clocked_accumulator.sv` 是教学 artifact。下一课把已知 neuron contract 拆成 combinational path + sequential register update。


## 13. 项目追踪 Project Trace

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- Teaching RTL: `rtl/learning/clocked_accumulator.sv`


## 14. Exit Ticket

你能解释 RTL / HDL / SystemVerilog / module / port，能读出一个最小 clocked module 的 state 和更新时刻，同时不会把“编译成功”误解成“功能验证通过”。
